# Lab 1 — Simulation HDFS avec Jupyter
## Atelier pratique : Commandes Hadoop Distributed File System

**Master Finance Digitale ISCE Casablanca**

---

### Objectifs
- ✅ Comprendre l'architecture HDFS (NameNode, DataNode)
- ✅ Exécuter les commandes HDFS essentielles
- ✅ Gérer des fichiers dans un système distribué
- ✅ Observer la réplication et la tolérance aux pannes

**Note :** Ce notebook simule HDFS. En production, vous utiliseriez un vrai cluster Hadoop (Docker, AWS EMR, etc.).

---

## Section 1 : Configuration du simulateur HDFS

Nous allons créer une **simulation légère de HDFS** en mémoire. Cela nous permettra d'exécuter les mêmes commandes que sur un vrai Hadoop.

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime
import shutil

# Créer un répertoire simulant HDFS
hdfs_root = Path("./hdfs_simulation")
hdfs_root.mkdir(exist_ok=True)

# Créer l'arborescence de base
user_dir = hdfs_root / "user" / "root"
user_dir.mkdir(parents=True, exist_ok=True)

print("✅ Simulateur HDFS initialisé")
print(f"📁 Répertoire racine : {hdfs_root.absolute()}")
print(f"👤 Utilisateur : root")
print(f"📍 Répertoire courant : /user/root")

## Section 2 : Commandes HDFS de base

Nous allons implémenter les commandes HDFS essentielles : `mkdir`, `ls`, `put`, `get`, `cat`, `rm`, etc.

In [ ]:
class HDFSSimulator:
    """Simulateur HDFS simple pour fins pédagogiques"""
    
    def __init__(self, root_path):
        self.root = Path(root_path)
        self.current_dir = self.root / "user" / "root"
        self.metadata = {}  # Métadonnées (propriétaire, permissions, etc.)
    
    def _get_path(self, path_str):
        """Résoudre un chemin relatif ou absolu"""
        if path_str.startswith("/"):
            return self.root / path_str.lstrip("/")
        else:
            return self.current_dir / path_str
    
    def mkdir(self, path):
        """Créer un répertoire (hdfs dfs -mkdir)"""
        p = self._get_path(path)
        p.mkdir(parents=True, exist_ok=True)
        return f"✅ Répertoire créé : {path}"
    
    def ls(self, path=".", recursive=False, human_readable=False):
        """Lister les fichiers (hdfs dfs -ls)"""
        p = self._get_path(path)
        
        if not p.exists():
            return f"❌ Erreur : {path} n'existe pas"
        
        items = []
        if recursive:
            for item in p.rglob("*"):
                rel_path = item.relative_to(self.root)
                size = item.stat().st_size if item.is_file() else 0
                size_str = self._format_size(size) if human_readable else size
                type_str = "d" if item.is_dir() else "-"
                items.append(f"{type_str}rw-r--r-- root root {size_str:>10} {rel_path}")
        else:
            for item in p.iterdir():
                rel_path = item.relative_to(self.root)
                size = item.stat().st_size if item.is_file() else 0
                size_str = self._format_size(size) if human_readable else size
                type_str = "d" if item.is_dir() else "-"
                items.append(f"{type_str}rw-r--r-- root root {size_str:>10} {rel_path}")
        
        return "\n".join(items) if items else "📂 Vide"
    
    def put(self, local_path, hdfs_path):
        """Copier un fichier local vers HDFS (hdfs dfs -put)"""
        local_p = Path(local_path)
        hdfs_p = self._get_path(hdfs_path)
        
        if not local_p.exists():
            return f"❌ Erreur : {local_path} n'existe pas"
        
        # Créer le répertoire parent si nécessaire
        hdfs_p.parent.mkdir(parents=True, exist_ok=True)
        
        shutil.copy(local_p, hdfs_p)
        size = hdfs_p.stat().st_size
        return f"✅ Fichier copié : {hdfs_path} ({self._format_size(size)})"
    
    def get(self, hdfs_path, local_path):
        """Télécharger un fichier de HDFS (hdfs dfs -get)"""
        hdfs_p = self._get_path(hdfs_path)
        local_p = Path(local_path)
        
        if not hdfs_p.exists():
            return f"❌ Erreur : {hdfs_path} n'existe pas dans HDFS"
        
        local_p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(hdfs_p, local_p)
        return f"✅ Fichier téléchargé : {local_path}"
    
    def cat(self, path):
        """Afficher le contenu d'un fichier (hdfs dfs -cat)"""
        p = self._get_path(path)
        
        if not p.exists():
            return f"❌ Erreur : {path} n'existe pas"
        
        if p.is_dir():
            return f"❌ Erreur : {path} est un répertoire"
        
        with open(p, 'r') as f:
            return f.read()
    
    def tail(self, path, lines=10):
        """Afficher les dernières lignes (hdfs dfs -tail)"""
        p = self._get_path(path)
        
        if not p.exists():
            return f"❌ Erreur : {path} n'existe pas"
        
        with open(p, 'r') as f:
            content = f.readlines()
        
        return ''.join(content[-lines:])
    
    def rm(self, path, recursive=False):
        """Supprimer un fichier ou répertoire (hdfs dfs -rm)"""
        p = self._get_path(path)
        
        if not p.exists():
            return f"❌ Erreur : {path} n'existe pas"
        
        if p.is_dir():
            if not recursive:
                return f"❌ Erreur : {path} est un répertoire (utilisez -r)"
            shutil.rmtree(p)
        else:
            p.unlink()
        
        return f"✅ Supprimé : {path}"
    
    def cp(self, src, dest):
        """Copier un fichier (hdfs dfs -cp)"""
        src_p = self._get_path(src)
        dest_p = self._get_path(dest)
        
        if not src_p.exists():
            return f"❌ Erreur : {src} n'existe pas"
        
        dest_p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src_p, dest_p)
        return f"✅ Copié : {src} → {dest}"
    
    def mv(self, src, dest):
        """Déplacer un fichier (hdfs dfs -mv)"""
        src_p = self._get_path(src)
        dest_p = self._get_path(dest)
        
        if not src_p.exists():
            return f"❌ Erreur : {src} n'existe pas"
        
        dest_p.parent.mkdir(parents=True, exist_ok=True)
        src_p.rename(dest_p)
        return f"✅ Déplacé : {src} → {dest}"
    
    def chmod(self, path, perms):
        """Changer les permissions (hdfs dfs -chmod)"""
        p = self._get_path(path)
        
        if not p.exists():
            return f"❌ Erreur : {path} n'existe pas"
        
        # Simulation simple
        return f"✅ Permissions changées : {path} → {perms}"
    
    def _format_size(self, size):
        """Formater la taille en Ko, Mo, Go"""
        for unit in ['B', 'KB', 'MB', 'GB']:
            if size < 1024:
                return f"{size:.1f}{unit}"
            size /= 1024
        return f"{size:.1f}TB"

# Créer l'instance du simulateur
hdfs = HDFSSimulator(hdfs_root)

print("✅ Simulateur HDFS initialisé")
print("Commandes disponibles : mkdir, ls, put, get, cat, tail, rm, cp, mv, chmod")

---

## Section 3 : Exercices pratiques — Reproduction de l'Atelier 1

Nous allons exécuter les mêmes commandes que dans l'atelier Docker.

### Exercice 1 : Créer l'arborescence de base

Commande : `hdfs dfs -mkdir -p /user/root`

In [ ]:
# Créer l'arborescence utilisateur
result = hdfs.mkdir("/user/root")
print(result)

# Vérifier
print("\n📂 Contenu du répertoire racine :")
print(hdfs.ls("/", recursive=True, human_readable=True))

### Exercice 2 : Créer un dossier 'input'

Commande : `hdfs dfs -mkdir input`

In [ ]:
result = hdfs.mkdir("input")
print(result)

print("\n📂 Contenu du répertoire courant (/user/root) :")
print(hdfs.ls())

### Exercice 3 : Créer un fichier de test

Simulons le téléchargement du fichier 'alice.txt' et sa copie vers HDFS.

In [ ]:
# Créer un fichier de test local (simulation de alice.txt)
test_file = Path("alice.txt")

# Contenu simplifié du conte d'Alice
alice_content = """Alice's Adventures in Wonderland
Lewis Carroll

Chapter I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do: once or twice she had peeped into the book her
sister was reading, but it had no pictures or conversations in it, 'and what
is the use of a book,' thought Alice 'without pictures or conversation?'

So she was considering in her own mind (as well as she could, for the hot
day made her feel very sleepy and stupid), whether the pleasure of making a
daisy-chain would be worth the trouble of getting up and picking the daisies,
when suddenly a White Rabbit with pink eyes ran close by her.

There was nothing so very remarkable in that; nor did Alice think it so very
much out of the way to hear the Rabbit say to itself, 'Oh dear! Oh dear! I
shall be late!' (when she thought it over afterwards, it occurred to her that
she ought to have wondered at this, but at the time it all seemed quite natural);
but when the Rabbit actually took a watch out of its waistcoat-pocket, and looked
at it, and then hurried on, Alice started to her feet, for it flashed across her
mind that she had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the field
after the White Rabbit, never considering how in the world she was to get in again.
"""

with open(test_file, 'w') as f:
    f.write(alice_content)

print(f"✅ Fichier créé : {test_file}")
print(f"📊 Taille : {test_file.stat().st_size} bytes")

### Exercice 4 : Copier le fichier vers HDFS

Commande : `hdfs dfs -put /local/alice.txt input/`

In [ ]:
result = hdfs.put("alice.txt", "input/alice.txt")
print(result)

print("\n📂 Contenu du dossier 'input' :")
print(hdfs.ls("input", human_readable=True))

### Exercice 5 : Afficher le contenu du fichier

Commande : `hdfs dfs -cat input/alice.txt`

In [ ]:
print("📄 Contenu du fichier :")
print("="*80)
content = hdfs.cat("input/alice.txt")
print(content[:500] + "...\n[tronqué pour lisibilité]")
print("="*80)

### Exercice 6 : Afficher les dernières lignes

Commande : `hdfs dfs -tail input/alice.txt`

In [ ]:
print("📄 Dernières lignes du fichier :")
print("="*80)
print(hdfs.tail("input/alice.txt", lines=5))
print("="*80)

### Exercice 7 : Copier le fichier dans une autre location

Commande : `hdfs dfs -cp input/alice.txt ./alice_copy.txt`

In [ ]:
result = hdfs.cp("input/alice.txt", "./alice_copy.txt")
print(result)

print("\n📂 Contenu du répertoire courant après copie :")
print(hdfs.ls(".", recursive=True, human_readable=True))

### Exercice 8 : Déplacer le fichier

Commande : `hdfs dfs -mv input/alice.txt ./alice_moved.txt`

In [ ]:
result = hdfs.mv("input/alice.txt", "./alice_moved.txt")
print(result)

print("\n📂 État après déplacement :")
print(hdfs.ls(".", recursive=True, human_readable=True))

### Exercice 9 : Télécharger le fichier localement

Commande : `hdfs dfs -get alice_moved.txt ./achat.txt`

In [ ]:
result = hdfs.get("./alice_moved.txt", "achat.txt")
print(result)

# Vérifier que le fichier existe
if Path("achat.txt").exists():
    print(f"✅ Fichier local créé : achat.txt ({Path('achat.txt').stat().st_size} bytes)")

### Exercice 10 : Supprimer des fichiers

Commande : `hdfs dfs -rm alice_copy.txt`

In [ ]:
result = hdfs.rm("./alice_copy.txt")
print(result)

print("\n📂 État final du système HDFS :")
print(hdfs.ls(".", recursive=True, human_readable=True))

---

## Section 4 : Concepts clés — Réplication et Tolérance aux pannes

En HDFS réel, chaque bloc est répliqué sur 3 nœuds différents.

In [ ]:
print("🏗️  Architecture HDFS (en production) :\n")

architecture = """
┌─────────────────────────────────────────┐
│         NameNode (maître)               │
│  - Gère l'arborescence des fichiers     │
│  - Ne stocke PAS les données            │
│  - Réplicate les blocs sur les DataNode │
└─────────────────────────────────────────┘
         │              │              │
         ▼              ▼              ▼
   ┌─────────┐    ┌─────────┐    ┌─────────┐
   │DataNode1│    │DataNode2│    │DataNode3│
   │ Bloc 1  │    │ Bloc 1  │    │ Bloc 1  │  ← Réplication ×3
   │ Bloc 2  │    │ Bloc 3  │    │ Bloc 2  │
   └─────────┘    └─────────┘    └─────────┘

Tolérance aux pannes :
- Si DataNode2 tombe → Bloc 1 et Bloc 3 existent ailleurs
- NameNode le détecte et réplique automatiquement sur un autre nœud
- Les données ne sont JAMAIS perdues
"""

print(architecture)

print("\n📊 Statistiques HDFS (simulation) :\n")
print(f"- Facteur de réplication : 3")
print(f"- Taille de bloc : 128 MB (configurable)")
print(f"- NameNode : stocke les métadonnées seulement")
print(f"- DataNodes : stockent les blocs de données")
print(f"- Heartbeat : 3 secondes (détection des pannes)")

---

## Section 5 : Synthèse et prochaines étapes

### Ce que vous avez appris

✅ **Commandes HDFS essentielles :**
- `hdfs dfs -mkdir` : créer des répertoires
- `hdfs dfs -ls` : lister les fichiers
- `hdfs dfs -put` : copier vers HDFS
- `hdfs dfs -get` : télécharger de HDFS
- `hdfs dfs -cat` : afficher le contenu
- `hdfs dfs -rm` : supprimer
- `hdfs dfs -cp` : copier
- `hdfs dfs -mv` : déplacer

✅ **Architecture HDFS :**
- NameNode : gère l'arborescence
- DataNode : stocke les blocs
- Réplication ×3 : tolérance aux pannes

✅ **Concepts clés :**
- Données distribuées sur plusieurs nœuds
- Redondance pour la fiabilité
- Scalabilité horizontale

### Prochaines étapes

1. **Lab 2 (Jour 1, Atelier 2) :** MapReduce et Spark sur Databricks
2. **Lab 3 (Jour 2) :** Spark SQL avancé et Machine Learning
3. **Atelier Docker (optionnel) :** Expérience avec un vrai cluster Hadoop

---

**Lab 1 complété !** 🎉

In [ ]:
print("""\n╔═══════════════════════════════════════════════════════╗
║  ✅ LAB 1 — SIMULATION HDFS COMPLÉTÉE                  ║
║                                                       ║
║  Vous avez maîtrisé les commandes HDFS essentielles  ║
║  et comprenez l'architecture distribuée.             ║
║                                                       ║
║  Passez au Lab 2 : MapReduce & Spark               ║
╚═══════════════════════════════════════════════════════╝
""")